# ComfyUI - Estúdio de Tradução e Lip-Sync HD (Colab Pro+)

Este notebook instala e inicia o **ComfyUI** com as ferramentas necessárias para você criar workflows visuais de tradução (Whisper), clonagem de voz e lip-sync avançado (MuseTalk/VideoReTalking).

**Importante:** Certifique-se de estar usando a GPU (A100 ou V100 recomendadas) acessando `Ambiente de Execução` -> `Alterar tipo de ambiente de execução`.

## Passo 1: Instalação do ComfyUI e ComfyUI-Manager

Baixando o núcleo do programa e o gerenciador de pacotes (Manager) que facilitará a instalação dos nós de áudio/vídeo.

In [ ]:
import os

# Clonar repositório do ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI

# Instalar as dependências base do ComfyUI
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt

# Instalar dependência essencial de vídeo (para carregar e salvar vídeos)
!pip install imageio-ffmpeg

# Clonar o ComfyUI-Manager na pasta de custom_nodes
%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ..

print("ComfyUI e Manager instalados com sucesso!")

## Passo 2: Instalar Cloudflared (Para acesso externo)

O ComfyUI roda localmente no servidor do Google. Precisamos criar um túnel seguro para que você possa acessar a interface pelo navegador no seu computador.

In [ ]:
# Baixa e instala o Cloudflare Tunnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("Cloudflared instalado com sucesso!")

## Passo 3: Iniciar o ComfyUI e Obter o Link

Rode a célula abaixo. Ele iniciará o servidor e o túnel. O código irá aguardar e imprimir um **Link de Acesso** (.trycloudflare.com) para você clicar e abrir a interface gráfica.

In [ ]:
import time
import re
import subprocess
import threading

print("Iniciando Servidor ComfyUI em segundo plano...")
# Inicia o servidor do ComfyUI
subprocess.Popen(["python", "main.py", "--dont-print-server"])

print("Iniciando túnel seguro do Cloudflare...")
# Inicia o túnel
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=open('cloudflare.log', 'w'), stderr=subprocess.STDOUT)

print("Aguardando a geração do link (isso pode levar até 20 segundos)...")

link_encontrado = False
for i in range(20):
    time.sleep(2)
    try:
        with open("cloudflare.log", "r") as f:
            conteudo = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", conteudo)
            if match:
                print("\n==================================================")
                print("✅ SUCESSO! CLIQUE NO LINK ABAIXO PARA ACESSAR:")
                print(match.group(0))
                print("==================================================\n")
                print("Mantenha esta aba do Colab aberta enquanto estiver usando o ComfyUI.")
                link_encontrado = True
                break
    except FileNotFoundError:
        pass

if not link_encontrado:
    print("O link demorou muito para ser gerado. Rode esta célula novamente!")

## Guia Rápido Pós-Acesso

Assim que você entrar na interface visual do ComfyUI pelo link gerado:

1. **Instalar os Nós de Áudio/Vídeo:**
   - No menu lateral, clique em **Manager**.
   - Clique em **Install Custom Nodes**.
   - Pesquise por `Whisper` (ex: ComfyUI-Audio/Whisper nodes) e instale.
   - Pesquise por `MuseTalk` ou `VideoReTalking` (ex: ComfyUI-MuseTalk) e instale.
   - Pesquise por opções de `TTS` (como XTTS ou Any-TTS custom nodes) e instale.
   - Reinicie o servidor do ComfyUI (reiniciando a célula 3 deste Colab) para que os nós sejam aplicados.

2. **Montar o Workflow:**
   - Adicione os nós na tela: `Load Video` -> `Whisper Translate` -> `TTS Generation` -> `MuseTalk Lip Sync` -> `Save Video`.
   - Conecte as saídas (bolinhas à direita) de áudio/texto de um nó com a entrada do próximo.
   - Pressione **Queue Prompt** para executar a tradução em altíssima qualidade!